# 🧠 Aceleración por Hardware (CUDA) para Modelos NLP

Este notebook de experimentación demuestra cómo configurar explícitamente el uso de **NVIDIA CUDA** para acelerar el entrenamiento e inferencia de modelos de clasificación usando la GPU local (ej. RTX 3050).

> **Estándar B2B:** No depender exclusivamente de CPU o APIs costosas en cloud cuando se dispone de hardware local premium. Aprovechar la GPU reduce tiempos de iteración sustancialmente.

In [ ]:
import torch
import logging
import warnings

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger()

## 1. Verificación del Hardware (Device Configuration)

El paso crítico es asegurar que PyTorch detecta los tensores en CUDA antes de instanciar el modelo.

In [ ]:
def setup_device():
    """
    Detecta y configura el dispositivo. Prioriza CUDA, luego MPS (Mac) y finalmente CPU.
    """
    if torch.cuda.is_available():
        device = torch.device('cuda')
        gpu_name = torch.cuda.get_device_name(0)
        logger.info(f"🚀 CUDA Detectado. Usando GPU: {gpu_name}")
        logger.info(f"  Memoria asignada: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
        logger.info(f"  Memoria cacheada: {torch.cuda.memory_reserved(0)/(1024**2):.2f} MB")
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        logger.info("🚀 Apple Silicon MPS Detectado. Usando GPU Mac.")
    else:
        device = torch.device('cpu')
        logger.warning("⚠️ No se detectó GPU. Cayendo a CPU (Esto será lento).")
        
    return device

device = setup_device()

## 2. Inferencia de prueba moviendo tensores a VRAM

A continuación, un ejemplo abstracto de cómo se instanciaría un modelo HuggingFace (ej. BAAI/bge-m3 o un clasificador bert) directamente a la tarjeta gráfica usando `.to(device)`.

In [ ]:
def mock_inference_demo():
    """
    Demostración de carga de tensores y variables a memoria VRAM.
    """
    # 1. Crear tensores dummy de gran tamaño en CPU
    tensor_a = torch.randn(10000, 10000)
    tensor_b = torch.randn(10000, 10000)
    
    # 2. Mover explícitamente a CUDA (.to)
    tensor_a_gpu = tensor_a.to(device)
    tensor_b_gpu = tensor_b.to(device)
    
    # 3. Operación de multiplicación de matrices acelerada
    # En CPU esto tardaría mucho, en RTX 3050 es casi instantáneo
    import time
    
    start_cpu = time.time()
    _ = torch.matmul(tensor_a, tensor_b)
    cpu_time = time.time() - start_cpu
    
    start_gpu = time.time()
    _ = torch.matmul(tensor_a_gpu, tensor_b_gpu)
    torch.cuda.synchronize() # Sincronizar para medir el tiempo real
    gpu_time = time.time() - start_gpu
    
    print(f"\n⏰ Rendimiento:")
    print(f"  Tiempo CPU: {cpu_time:.4f}s")
    print(f"  Tiempo GPU: {gpu_time:.4f}s")
    print(f"  ⚡ Aceleración de {cpu_time/gpu_time:.1f}x veces más rápido.")

if device.type == 'cuda':
    mock_inference_demo()

## 3. Próximos Pasos (Arquitectura Microservicio)

Los pipelines probados aquí (ej. extracciones de embeddings o re-entrenamiento con LoRA para ajustar a normativas DIAN) deben exportarse al directorio `src/models/` y exponerse a través de `api/main.py`. Asegúrate de eliminar variables innecesarias con `del variable` y limpiar la VRAM con `torch.cuda.empty_cache()` en endpoints de larga ejecución.